<a href="https://colab.research.google.com/github/Simmartie/Operations-strategie/blob/main/DEA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#dataset opladen
import pandas as pd
from google.colab import files

# Upload het Excel-bestand
uploaded = files.upload()

# Laad de dataset in een DataFrame
# Note: Pas eventueel de bestandsnaam aan als de extensie .xlsx of .xls is
df = pd.read_excel('dataset_final.xlsx')
display(df.head())

Saving dataset_final.xlsx to dataset_final.xlsx


,Company,Net PP&E (mm),COGS (mm),SG&A (mm),OPEX (mm),Operational power capacity (MW),Revenue (mm),EBITDA (mm)
0,Digital Core Reit 2024,1852.0,40.400,10.200,50.600,63.893,102.300,330.593
1,Digital Core Reit 2025,2003.9,87.400,14.800,102.200,59.100,176.200,109.443
2,Digital Realty Trust Inc 2023,26532.0,4418.822,533.778,4952.600,2500.000,5515.701,2219.400
3,Digital Realty Trust Inc 2024,26760.6,4509.179,573.925,5083.104,2700.000,5589.073,2243.700
4,Digital Realty Trust Inc 2025,29861.5,4703.628,750.572,5454.200,2900.000,6241.757,2553.100


In [ ]:
import numpy as np
import pandas as pd

# Data opschonen (lege rijen verwijderen en filteren op de relevante DMUs)
df_clean = df.dropna(subset=['Company', 'Net PP&E (mm)', 'OPEX (mm)', 'Operational power capacity (MW)', 'Revenue (mm)', 'EBITDA (mm)']).copy()

# DMU namen ophalen
dmus = df_clean['Company'].tolist()

# Inputs (X) en Outputs (Y)
# We hebben 3 inputs en 2 outputs
X = df_clean[['Net PP&E (mm)', 'OPEX (mm)', 'Operational power capacity (MW)']].values
Y = df_clean[['Revenue (mm)', 'EBITDA (mm)']].values

print(f"Aantal DMUs geïdentificeerd: {len(dmus)}")
display(df_clean[['Company', 'Net PP&E (mm)', 'OPEX (mm)', 'Operational power capacity (MW)', 'Revenue (mm)', 'EBITDA (mm)']].head(20))

Aantal DMUs geïdentificeerd: 17


,Company,Net PP&E (mm),OPEX (mm),Operational power capacity (MW),Revenue (mm),EBITDA (mm)
0,Digital Core Reit 2024,1852.000,50.600,63.893,102.300,330.593
1,Digital Core Reit 2025,2003.900,102.200,59.100,176.200,109.443
2,Digital Realty Trust Inc 2023,26532.000,4952.600,2500.000,5515.701,2219.400
3,Digital Realty Trust Inc 2024,26760.600,5083.104,2700.000,5589.073,2243.700
4,Digital Realty Trust Inc 2025,29861.500,5454.200,2900.000,6241.757,2553.100
5,Equinix Inc 2023,20049.723,4886.041,1812.500,8188.136,3302.095
6,Equinix Inc 2024,20668.000,5109.000,1857.000,8748.000,3639.000
7,Equinix Inc 2025,24976.000,5253.000,1961.500,9217.000,3964.000
8,Keppel Dc Reit 2023,2778.000,50.500,306.600,209.400,138.700
9,Keppel Dc Reit 2024,3599.800,67.700,318.300,232.200,269.900


In [ ]:
# Importeer de benodigde wiskundige optimalisatiefuncties en bibliotheken
from scipy.optimize import minimize, LinearConstraint
from scipy.optimize import milp, Bounds
from scipy.optimize import linprog
import numpy as np
import pandas as pd

# --- Uitleg Output-Georiënteerd DEA Model ---
# Doel: Maximaliseer de output-expansiefactor 'phi' (waarbij phi >= 1).
# Een score van 1.0 (of phi = 1.0) betekent dat de DMU volledig efficiënt is.
# We zoeken de optimale gewichten (lambdas) van de referentiegroep (peers).
# Restricties:
# 1. De virtuele input van de referentiegroep mag de werkelijke input van DMU_k niet overschrijden.
# 2. De virtuele output van de referentiegroep moet minstens gelijk zijn aan phi maal de output van DMU_k.
# 3. Alle lambda-gewichten moeten groter dan of gelijk aan 0 zijn.
# 4. Voor VRS (BCC) voegen we de extra restrictie toe dat de som van de lambdas exact 1 moet zijn.

# Bepaal het aantal DMUs, inputs en outputs op basis van onze geladen matrices
n_dmus = len(dmus)
n_inputs = X.shape[1]
n_outputs = Y.shape[1]

# Maak lege lijsten om de berekende efficiëntiescores voor CRS en VRS in op te slaan
scores_crs = []
scores_vrs = []

# Loop door elke DMU (bedrijf) om de individuele efficiëntie te berekenen
for k in range(n_dmus):
    # We definiëren de beslissingsvariabelen vector: [phi, lambda_0, lambda_1, ..., lambda_n]
    # phi staat op index 0, de lambda-gewichten voor alle DMUs starten vanaf index 1

    # scipy.optimize.linprog minimaliseert standaard.
    # Omdat we phi willen maximaliseren, minimaliseren we (-1.0 * phi).
    # De doelfunctiecoëfficiënten worden dus: [-1.0, 0, 0, ..., 0]
    c = np.zeros(1 + n_dmus)
    c[0] = -1.0

    # --- INPUT RESTRICTIES (A_ub * variabelen <= b_ub) ---
    # Formule: sum(lambda_i * X_i) <= X_k  =>  0 * phi + sum(lambda_i * X_i) <= X_k
    # We maken een matrix met rij-grootte gelijk aan het aantal inputs
    A_inputs = np.zeros((n_inputs, 1 + n_dmus))
    # Vul de kolommen vanaf index 1 met de getransponeerde inputmatrix van alle DMUs
    A_inputs[:, 1:] = X.T
    # De rechterzijde van de vergelijking (b) is de inputvector van de huidige DMU_k
    b_inputs = X[k, :]

    # --- OUTPUT RESTRICTIES ---
    # Formule: phi * Y_k <= sum(lambda_i * Y_i)  =>  phi * Y_k - sum(lambda_i * Y_i) <= 0
    # We maken een matrix met rij-grootte gelijk aan het aantal outputs
    A_outputs = np.zeros((n_outputs, 1 + n_dmus))
    # De coëfficiënt voor phi (index 0) is de output van de huidige DMU_k
    A_outputs[:, 0] = Y[k, :]
    # De coëfficiënten voor de lambdas (index 1 t/m n) zijn de negatieve outputs van alle DMUs
    A_outputs[:, 1:] = -Y.T
    # De rechterzijde van deze ongelijkheid is 0
    b_outputs = np.zeros(n_outputs)

    # Voeg de input- en outputrestricties samen tot één grote matrix en vector
    A_ub = np.vstack([A_inputs, A_outputs])
    b_ub = np.concatenate([b_inputs, b_outputs])

    # Definieer de grenzen (bounds) voor de variabelen:
    # phi moet groter of gelijk zijn aan 1.0; lambdas moeten groter of gelijk zijn aan 0.0
    bounds = [(1.0, None)] + [(0.0, None)] * n_dmus

    # --- BEREKENING CRS (CCR) MODEL ---
    # Los het lineaire programmeringsprobleem op met de 'highs' solver
    res_crs = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    # Als de optimalisatie is geslaagd, berekenen we de efficiëntie als (1 / phi)
    if res_crs.success:
        scores_crs.append(1.0 / res_crs.x[0])
    else:
        scores_crs.append(np.nan)

    # --- BEREKENING VRS (BCC) MODEL ---
    # Voeg de gelijkheidsrestrictie toe: de som van alle lambdas moet gelijk zijn aan 1
    # Formule: 0 * phi + sum(lambda_i) = 1
    A_eq = np.zeros((1, 1 + n_dmus))
    # Zet de coëfficiënten voor alle lambdas (index 1 t/m n) op 1.0
    A_eq[0, 1:] = 1.0
    # De rechterzijde van de vergelijking is exact 1.0
    b_eq = np.array([1.0])

    # Los het VRS LP-probleem op inclusief de gelijkheidsrestrictie
    res_vrs = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    # Als de optimalisatie is geslaagd, berekenen we de efficiëntie als (1 / phi)
    if res_vrs.success:
        scores_vrs.append(1.0 / res_vrs.x[0])
    else:
        scores_vrs.append(np.nan)

# Combineer de resultaten en de namen van de DMUs (bedrijven) in een overzichtelijke DataFrame
results_df = pd.DataFrame({
    'DMU (Company)': dmus,
    'CRS Efficiency (CCR)': scores_crs,
    'VRS Efficiency (BCC)': scores_vrs
})

# Rond de berekende efficiëntiescores af op 4 decimalen voor een nette weergave
results_df['CRS Efficiency (CCR)'] = results_df['CRS Efficiency (CCR)'].round(4)
results_df['VRS Efficiency (BCC)'] = results_df['VRS Efficiency (BCC)'].round(4)

# Toon de definitieve tabel met efficiëntiescores op het scherm
display(results_df)

,DMU (Company),CRS Efficiency (CCR),VRS Efficiency (BCC)
0,Digital Core Reit 2024,1.0000,1.0000
1,Digital Core Reit 2025,0.9058,1.0000
2,Digital Realty Trust Inc 2023,0.6245,0.6339
3,Digital Realty Trust Inc 2024,0.6181,0.6262
4,Digital Realty Trust Inc 2025,0.6397,0.6772
5,Equinix Inc 2023,0.9767,0.9769
6,Equinix Inc 2024,1.0000,1.0000
7,Equinix Inc 2025,1.0000,1.0000
8,Keppel Dc Reit 2023,1.0000,1.0000
9,Keppel Dc Reit 2024,0.8922,0.9090


In [ ]:
# Bereken de Scale Efficiency (SE) als verhouding tussen CRS (CCR) en VRS (BCC) score
# Formule: Scale Efficiency = CRS_score / VRS_score
results_df['Scale Efficiency'] = (results_df['CRS Efficiency (CCR)'] / results_df['VRS Efficiency (BCC)']).round(4)

# Bepaal het type schaalopbrengst (IRS/DRS/MPSS) kan eventueel ook, maar we tonen eerst de score
# Toon de bijgewerkte tabel met CRS, VRS en Scale Efficiency
display(results_df)

,DMU (Company),CRS Efficiency (CCR),VRS Efficiency (BCC),Scale Efficiency
0,Digital Core Reit 2024,1.0000,1.0000,1.0000
1,Digital Core Reit 2025,0.9058,1.0000,0.9058
2,Digital Realty Trust Inc 2023,0.6245,0.6339,0.9852
3,Digital Realty Trust Inc 2024,0.6181,0.6262,0.9871
4,Digital Realty Trust Inc 2025,0.6397,0.6772,0.9446
5,Equinix Inc 2023,0.9767,0.9769,0.9998
6,Equinix Inc 2024,1.0000,1.0000,1.0000
7,Equinix Inc 2025,1.0000,1.0000,1.0000
8,Keppel Dc Reit 2023,1.0000,1.0000,1.0000
9,Keppel Dc Reit 2024,0.8922,0.9090,0.9815


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Zorg ervoor dat de data correct is geladen en gedefinieerd
df_clean = df.dropna(subset=['Company', 'Net PP&E (mm)', 'OPEX (mm)', 'Operational power capacity (MW)', 'Revenue (mm)', 'EBITDA (mm)']).copy()
dmus = df_clean['Company'].tolist()

X = df_clean[['Net PP&E (mm)', 'OPEX (mm)', 'Operational power capacity (MW)']].values
Y = df_clean[['Revenue (mm)', 'EBITDA (mm)']].values

n_dmus = len(dmus)
n_inputs = X.shape[1]
n_outputs = Y.shape[1]

# We initialiseren matrices om de lambdas in op te slaan
lambdas_crs = np.zeros((n_dmus, n_dmus))
lambdas_vrs = np.zeros((n_dmus, n_dmus))

for k in range(n_dmus):
    # --- CRS Model ---
    c = np.zeros(1 + n_dmus)
    c[0] = -1.0

    A_inputs = np.zeros((n_inputs, 1 + n_dmus))
    A_inputs[:, 1:] = X.T
    b_inputs = X[k, :]

    A_outputs = np.zeros((n_outputs, 1 + n_dmus))
    A_outputs[:, 0] = Y[k, :]
    A_outputs[:, 1:] = -Y.T
    b_outputs = np.zeros(n_outputs)

    A_ub = np.vstack([A_inputs, A_outputs])
    b_ub = np.concatenate([b_inputs, b_outputs])

    bounds = [(1.0, None)] + [(0.0, None)] * n_dmus

    res_crs = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    if res_crs.success:
        # Sla de lambdas op
        lambdas_crs[k, :] = res_crs.x[1:]

    # --- VRS Model ---
    A_eq = np.zeros((1, 1 + n_dmus))
    A_eq[0, 1:] = 1.0
    b_eq = np.array([1.0])

    res_vrs = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    if res_vrs.success:
        # Sla de lambdas op
        lambdas_vrs[k, :] = res_vrs.x[1:]

# Maak DataFrames voor de weergave van de lambdas
df_lambdas_crs = pd.DataFrame(lambdas_crs, columns=[f'L_{name}' for name in dmus], index=dmus).round(4)
df_lambdas_vrs = pd.DataFrame(lambdas_vrs, columns=[f'L_{name}' for name in dmus], index=dmus).round(4)

print("--- GEREGISTREERDE LAMBDAS (PEER WEIGHTS) ONDER CRS (CCR) ---")
display(df_lambdas_crs)

print("\n--- GEREGISTREERDE LAMBDAS (PEER WEIGHTS) ONDER VRS (BCC) ---")
display(df_lambdas_vrs)

--- GEREGISTREERDE LAMBDAS (PEER WEIGHTS) ONDER CRS (CCR) ---


,L_Digital Core Reit 2024,L_Digital Core Reit 2025,L_Digital Realty Trust Inc 2023,L_Digital Realty Trust Inc 2024,L_Digital Realty Trust Inc 2025,L_Equinix Inc 2023,L_Equinix Inc 2024,L_Equinix Inc 2025,L_Keppel Dc Reit 2023,L_Keppel Dc Reit 2024,L_Keppel Dc Reit 2025,L_Nextdc Ltd 2023,L_Nextdc Ltd 2024,L_Nextdc Ltd 2025,L_GDS Holdings 2023,L_GDS Holdings 2024,L_GDS Holdings 2025
Digital Core Reit 2024,1.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
Digital Core Reit 2025,0.0657,0.0,0.0,0.0,0.0,0.0,0.0000,0.0176,0.0000,0.0,0.0746,0.0,0.0,0.0,0.0,0.0,0.0
Digital Realty Trust Inc 2023,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.9315,1.1759,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
Digital Realty Trust Inc 2024,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.9578,1.0215,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
Digital Realty Trust Inc 2025,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,1.0234,1.5481,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
Equinix Inc 2023,0.0000,0.0,0.0,0.0,0.0,0.0,0.8781,0.0761,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
Equinix Inc 2024,0.0000,0.0,0.0,0.0,0.0,0.0,1.0000,0.0000,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
Equinix Inc 2025,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
Keppel Dc Reit 2023,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,-0.0000,1.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
Keppel Dc Reit 2024,0.0352,0.0,0.0,0.0,0.0,0.0,0.0000,0.0009,0.0000,0.0,0.7347,0.0,0.0,0.0,0.0,0.0,0.0



--- GEREGISTREERDE LAMBDAS (PEER WEIGHTS) ONDER VRS (BCC) ---


,L_Digital Core Reit 2024,L_Digital Core Reit 2025,L_Digital Realty Trust Inc 2023,L_Digital Realty Trust Inc 2024,L_Digital Realty Trust Inc 2025,L_Equinix Inc 2023,L_Equinix Inc 2024,L_Equinix Inc 2025,L_Keppel Dc Reit 2023,L_Keppel Dc Reit 2024,L_Keppel Dc Reit 2025,L_Nextdc Ltd 2023,L_Nextdc Ltd 2024,L_Nextdc Ltd 2025,L_GDS Holdings 2023,L_GDS Holdings 2024,L_GDS Holdings 2025
Digital Core Reit 2024,1.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,-0.0000,0.0000,0.0,-0.0000,0.0000,0.0,0.0,0.0,0.0,0.0
Digital Core Reit 2025,0.0000,1.0000,0.0,0.0,0.0,0.0,0.0000,-0.0000,0.0000,0.0,-0.0000,0.0000,0.0,0.0,0.0,0.0,0.0
Digital Realty Trust Inc 2023,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.9419,0.0000,0.0,0.0581,0.0000,0.0,0.0,0.0,0.0,0.0
Digital Realty Trust Inc 2024,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.9671,0.0000,0.0,0.0329,0.0000,0.0,0.0,0.0,0.0,0.0
Digital Realty Trust Inc 2025,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0
Equinix Inc 2023,0.0000,0.0000,0.0,0.0,0.0,0.0,0.9098,0.0448,0.0454,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0
Equinix Inc 2024,0.0000,0.0000,0.0,0.0,0.0,0.0,1.0000,0.0000,0.0000,0.0,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0
Equinix Inc 2025,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0,-0.0000,0.0000,0.0,0.0,0.0,0.0,0.0
Keppel Dc Reit 2023,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,1.0000,0.0,-0.0000,0.0000,0.0,0.0,0.0,0.0,0.0
Keppel Dc Reit 2024,0.1764,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0002,0.3339,0.0,0.4895,0.0000,0.0,0.0,0.0,0.0,0.0


In [ ]:
# --- ROBUUSTHEIDSCHECK: DEA-model zonder OPEX ---
# Inputs: 'Net PP&E (mm)' en 'Operational power capacity (MW)'
# Outputs: 'Revenue (mm)' en 'EBITDA (mm)'

X_robust = df_clean[['Net PP&E (mm)', 'Operational power capacity (MW)']].values
Y_robust = Y # Dezelfde outputs

n_inputs_robust = X_robust.shape[1]
scores_crs_robust = []
scores_vrs_robust = []

for k in range(n_dmus):
    # Doelfunctiecoëfficiënten voor maximalisatie van phi (geïmplementeerd als minimalisatie van -phi)
    c = np.zeros(1 + n_dmus)
    c[0] = -1.0

    # Input restricties (zonder OPEX)
    A_inputs = np.zeros((n_inputs_robust, 1 + n_dmus))
    A_inputs[:, 1:] = X_robust.T
    b_inputs = X_robust[k, :]

    # Output restricties
    A_outputs = np.zeros((n_outputs, 1 + n_dmus))
    A_outputs[:, 0] = Y_robust[k, :]
    A_outputs[:, 1:] = -Y_robust.T
    b_outputs = np.zeros(n_outputs)

    # Samenvoegen van restricties
    A_ub = np.vstack([A_inputs, A_outputs])
    b_ub = np.concatenate([b_inputs, b_outputs])

    # Variabele grenzen
    bounds = [(1.0, None)] + [(0.0, None)] * n_dmus

    # --- CRS Model ---
    res_crs = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    if res_crs.success:
        scores_crs_robust.append(1.0 / res_crs.x[0])
    else:
        scores_crs_robust.append(np.nan)

    # --- VRS Model ---
    A_eq = np.zeros((1, 1 + n_dmus))
    A_eq[0, 1:] = 1.0
    b_eq = np.array([1.0])

    res_vrs = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    if res_vrs.success:
        scores_vrs_robust.append(1.0 / res_vrs.x[0])
    else:
        scores_vrs_robust.append(np.nan)

# Resultaten compileren
robust_df = pd.DataFrame({
    'DMU (Company)': dmus,
    'CRS (No OPEX)': scores_crs_robust,
    'VRS (No OPEX)': scores_vrs_robust
})

# Bereken Scale Efficiency voor het robuuste model
robust_df['Scale Efficiency (No OPEX)'] = (robust_df['CRS (No OPEX)'] / robust_df['VRS (No OPEX)']).round(4)

# Rond overige scores af
robust_df['CRS (No OPEX)'] = robust_df['CRS (No OPEX)'].round(4)
robust_df['VRS (No OPEX)'] = robust_df['VRS (No OPEX)'].round(4)

display(robust_df)

,DMU (Company),CRS (No OPEX),VRS (No OPEX),Scale Efficiency (No OPEX)
0,Digital Core Reit 2024,1.0000,1.0000,1.0000
1,Digital Core Reit 2025,0.7182,1.0000,0.7182
2,Digital Realty Trust Inc 2023,0.4912,0.5984,0.8207
3,Digital Realty Trust Inc 2024,0.4934,0.6064,0.8137
4,Digital Realty Trust Inc 2025,0.4938,0.6772,0.7292
5,Equinix Inc 2023,0.9649,0.9670,0.9978
6,Equinix Inc 2024,1.0000,1.0000,1.0000
7,Equinix Inc 2025,1.0000,1.0000,1.0000
8,Keppel Dc Reit 2023,0.2819,0.3593,0.7846
9,Keppel Dc Reit 2024,0.4216,0.4231,0.9963


In [ ]:
# --- ROBUUSTHEIDSCHECK 2: DEA-model met OPEX, maar zonder MW ---
# Inputs: 'Net PP&E (mm)' en 'OPEX (mm)'
# Outputs: 'Revenue (mm)' en 'EBITDA (mm)'

X_no_mw = df_clean[['Net PP&E (mm)', 'OPEX (mm)']].values
Y_no_mw = Y # Dezelfde outputs

n_inputs_no_mw = X_no_mw.shape[1]
scores_crs_no_mw = []
scores_vrs_no_mw = []

for k in range(n_dmus):
    # Doelfunctiecoëfficiënten voor maximalisatie van phi
    c = np.zeros(1 + n_dmus)
    c[0] = -1.0

    # Input restricties (zonder MW)
    A_inputs = np.zeros((n_inputs_no_mw, 1 + n_dmus))
    A_inputs[:, 1:] = X_no_mw.T
    b_inputs = X_no_mw[k, :]

    # Output restricties
    A_outputs = np.zeros((n_outputs, 1 + n_dmus))
    A_outputs[:, 0] = Y_no_mw[k, :]
    A_outputs[:, 1:] = -Y_no_mw.T
    b_outputs = np.zeros(n_outputs)

    # Samenvoegen van restricties
    A_ub = np.vstack([A_inputs, A_outputs])
    b_ub = np.concatenate([b_inputs, b_outputs])

    # Variabele grenzen
    bounds = [(1.0, None)] + [(0.0, None)] * n_dmus

    # --- CRS Model ---
    res_crs = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    if res_crs.success:
        scores_crs_no_mw.append(1.0 / res_crs.x[0])
    else:
        scores_crs_no_mw.append(np.nan)

    # --- VRS Model ---
    A_eq = np.zeros((1, 1 + n_dmus))
    A_eq[0, 1:] = 1.0
    b_eq = np.array([1.0])

    res_vrs = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    if res_vrs.success:
        scores_vrs_no_mw.append(1.0 / res_vrs.x[0])
    else:
        scores_vrs_no_mw.append(np.nan)

# Resultaten compileren
no_mw_df = pd.DataFrame({
    'DMU (Company)': dmus,
    'CRS (No MW)': scores_crs_no_mw,
    'VRS (No MW)': scores_vrs_no_mw
})

# Bereken Scale Efficiency voor dit model
no_mw_df['Scale Efficiency (No MW)'] = (no_mw_df['CRS (No MW)'] / no_mw_df['VRS (No MW)']).round(4)

# Rond overige scores af
no_mw_df['CRS (No MW)'] = no_mw_df['CRS (No MW)'].round(4)
no_mw_df['VRS (No MW)'] = no_mw_df['VRS (No MW)'].round(4)

display(no_mw_df)

,DMU (Company),CRS (No MW),VRS (No MW),Scale Efficiency (No MW)
0,Digital Core Reit 2024,1.0000,1.0000,1.0000
1,Digital Core Reit 2025,0.7058,0.7898,0.8936
2,Digital Realty Trust Inc 2023,0.6245,0.6339,0.9852
3,Digital Realty Trust Inc 2024,0.6181,0.6262,0.9871
4,Digital Realty Trust Inc 2025,0.6397,0.6772,0.9447
5,Equinix Inc 2023,0.9767,0.9769,0.9998
6,Equinix Inc 2024,1.0000,1.0000,1.0000
7,Equinix Inc 2025,1.0000,1.0000,1.0000
8,Keppel Dc Reit 2023,1.0000,1.0000,1.0000
9,Keppel Dc Reit 2024,0.8922,0.9090,0.9815


In [ ]:
# --- DEA-model zonder EBITDA ---
# Inputs: 'Net PP&E (mm)', 'OPEX (mm)' en 'Operational power capacity (MW)' (ongewijzigd)
# Outputs: Alleen 'Revenue (mm)'

# Filter de outputmatrix naar alleen de eerste kolom (Revenue)
Y_no_ebitda = Y[:, [0]]

n_outputs_no_ebitda = Y_no_ebitda.shape[1]
scores_crs_no_ebitda = []
scores_vrs_no_ebitda = []

for k in range(n_dmus):
    # Doelfunctiecofficinten voor maximalisatie van phi
    c = np.zeros(1 + n_dmus)
    c[0] = -1.0

    # Input restricties
    A_inputs = np.zeros((n_inputs, 1 + n_dmus))
    A_inputs[:, 1:] = X.T
    b_inputs = X[k, :]

    # Output restricties (alleen Revenue)
    A_outputs = np.zeros((n_outputs_no_ebitda, 1 + n_dmus))
    A_outputs[:, 0] = Y_no_ebitda[k, :]
    A_outputs[:, 1:] = -Y_no_ebitda.T
    b_outputs = np.zeros(n_outputs_no_ebitda)

    # Samenvoegen van restricties
    A_ub = np.vstack([A_inputs, A_outputs])
    b_ub = np.concatenate([b_inputs, b_outputs])

    # Variabele grenzen
    bounds = [(1.0, None)] + [(0.0, None)] * n_dmus

    # --- CRS Model ---
    res_crs = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    if res_crs.success:
        scores_crs_no_ebitda.append(1.0 / res_crs.x[0])
    else:
        scores_crs_no_ebitda.append(np.nan)

    # --- VRS Model ---
    A_eq = np.zeros((1, 1 + n_dmus))
    A_eq[0, 1:] = 1.0
    b_eq = np.array([1.0])

    res_vrs = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    if res_vrs.success:
        scores_vrs_no_ebitda.append(1.0 / res_vrs.x[0])
    else:
        scores_vrs_no_ebitda.append(np.nan)

# Resultaten compileren in een DataFrame
no_ebitda_df = pd.DataFrame({
    'DMU (Company)': dmus,
    'CRS (No EBITDA)': scores_crs_no_ebitda,
    'VRS (No EBITDA)': scores_vrs_no_ebitda
})

# Bereken Scale Efficiency voor dit model
no_ebitda_df['Scale Efficiency (No EBITDA)'] = (no_ebitda_df['CRS (No EBITDA)'] / no_ebitda_df['VRS (No EBITDA)']).round(4)

# Rond overige scores af
no_ebitda_df['CRS (No EBITDA)'] = no_ebitda_df['CRS (No EBITDA)'].round(4)
no_ebitda_df['VRS (No EBITDA)'] = no_ebitda_df['VRS (No EBITDA)'].round(4)

print("--- DEA EFFICINTIESCORES (ZONDER EBITDA OMZET ALS ENIGE OUTPUT) ---")
display(no_ebitda_df)

--- DEA EFFICINTIESCORES (ZONDER EBITDA OMZET ALS ENIGE OUTPUT) ---


,DMU (Company),CRS (No EBITDA),VRS (No EBITDA),Scale Efficiency (No EBITDA)
0,Digital Core Reit 2024,0.8209,1.0000,0.8209
1,Digital Core Reit 2025,0.8990,1.0000,0.8990
2,Digital Realty Trust Inc 2023,0.6245,0.6339,0.9852
3,Digital Realty Trust Inc 2024,0.6181,0.6262,0.9871
4,Digital Realty Trust Inc 2025,0.6397,0.6772,0.9447
5,Equinix Inc 2023,0.9767,0.9769,0.9998
6,Equinix Inc 2024,1.0000,1.0000,1.0000
7,Equinix Inc 2025,1.0000,1.0000,1.0000
8,Keppel Dc Reit 2023,1.0000,1.0000,1.0000
9,Keppel Dc Reit 2024,0.8611,0.8662,0.9941
